In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# Load all three splits
print("\nLoading data...")
X_train = pd.read_csv('../extracted/X_train.csv')
X_val   = pd.read_csv('../extracted/X_val.csv')
X_test  = pd.read_csv('../extracted/X_test.csv')
y_train = pd.read_csv('../extracted/y_train_binary.csv').squeeze()
y_val   = pd.read_csv('../extracted/y_val_binary.csv').squeeze()
y_test  = pd.read_csv('../extracted/y_test_binary.csv').squeeze()

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

# Convert to numpy
X_train_np = X_train.values.astype(np.float32)
X_val_np   = X_val.values.astype(np.float32)
X_test_np  = X_test.values.astype(np.float32)
y_train_np = y_train.values.astype(np.float32)
y_val_np   = y_val.values.astype(np.float32)
y_test_np  = y_test.values.astype(np.float32)

# Class weights
neg, pos = np.bincount(y_train_np.astype(int))
class_weight = {0: 5.0, 1: 1.0}
print(f"\nClass weight: {class_weight}")
print("\nReady.")

TensorFlow: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Loading data...
Train: (14703605, 39)
Val:   (2100516, 39)
Test:  (4201031, 39)

Class weight: {0: 5.0, 1: 1.0}

Ready.


In [9]:
# === MODEL 1: MLP ===
print("Building MLP...")

mlp = keras.Sequential([
    layers.Input(shape=(39,)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
], name='MLP')

optimizer = keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)
mlp.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

mlp.summary()

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("\nTraining MLP...")
t0 = time.time()

history_mlp = mlp.fit(
    X_train_np, y_train_np,
    epochs=10,
    batch_size=4096,
    validation_data=(X_val_np, y_val_np),
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

mlp_time = time.time() - t0
print(f"\nTraining time: {mlp_time:.1f}s")

Building MLP...


Model: "MLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 256)            │        10,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,993 (207.00 KB)

 Trainable params: 52,225 (204.00 KB)

 Non-trainable params: 768 (3.00 KB)


Training MLP...
Epoch 1/10
3590/3590 ━━━━━━━━━━━━━━━━━━━━ 62s 17ms/step - accuracy: 0.8677 - loss: 0.3901 - val_accuracy: 0.9582 - val_loss: 0.1016
Epoch 2/10
3590/3590 ━━━━━━━━━━━━━━━━━━━━ 62s 17ms/step - accuracy: 0.9584 - loss: 0.1640 - val_accuracy: 0.9566 - val_loss: 0.1006
Epoch 3/10
3590/3590 ━━━━━━━━━━━━━━━━━━━━ 61s 17ms/step - accuracy: 0.9600 - loss: 0.1458 - val_accuracy: 0.9588 - val_loss: 0.0944
Epoch 4/10
3590/3590 ━━━━━━━━━━━━━━━━━━━━ 60s 17ms/step - accuracy: 0.9603 - loss: 0.1412 - val_accuracy: 0.9573 - val_loss: 0.0938
Epoch 5/10
1997/3590 ━━━━━━━━━━━━━━━━━━━━ 26s 17ms/step - accuracy: 0.9603 - loss: 0.1400

KeyboardInterrupt: 